# 🚀 Building RAG from Scratch - Learning Notebook

**Learn Retrieval-Augmented Generation (RAG) Step by Step**

This notebook teaches you how to build a RAG system from the ground up using only basic Python libraries:
- **OpenAI** for embeddings and text generation
- **NumPy** for vector similarity calculations  
- **JSON** for simple vector database storage

## 📚 What You'll Learn:

1. **Document Processing** - Loading and converting JSON to text
2. **Text Chunking** - Breaking documents into manageable pieces  
3. **Embeddings** - Converting text to numerical vectors
4. **Vector Search** - Finding similar content using cosine similarity
5. **Answer Generation** - Using AI to create contextual responses
6. **Transparent Outputs** - Saving all intermediate results to files

## 🎯 Goal: 
Build a working RAG system that can answer questions about the **Flintstone API** documentation!

---

## 📦 Step 1: Import Libraries & Configuration

First, let's import all the libraries we need and set up our configuration.

In [ ]:
import json
import os
import numpy as np
from typing import List, Dict, Any
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# ✅ Configuration - All settings in one place
class RAGConfig:
    # OpenAI Settings
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    GENERATION_MODEL = "gpt-3.5-turbo" 
    EMBEDDING_MODEL = "text-embedding-ada-002"
    TEMPERATURE = 0.1
    MAX_TOKENS = 2000
    
    # Document Processing
    CHUNK_SIZE = 1000
    CHUNK_OVERLAP = 200
    
    # Files
    INPUT_DOCUMENT = "./documents/flintstone_api.json"
    VECTOR_DATABASE_FILE = "vector_database.json"
    
    # RAG Settings
    TOP_K_RESULTS = 3

# Initialize OpenAI client
if not RAGConfig.OPENAI_API_KEY:
    raise ValueError("❌ Please set OPENAI_API_KEY in your .env file")

client = OpenAI(api_key=RAGConfig.OPENAI_API_KEY)

print("✅ Libraries imported and configuration loaded!")
print(f"📁 Input document: {RAGConfig.INPUT_DOCUMENT}")
print(f"🗄️ Vector database: {RAGConfig.VECTOR_DATABASE_FILE}")
print(f"🔢 Chunk size: {RAGConfig.CHUNK_SIZE} characters")
print(f"🔍 Retrieving top {RAGConfig.TOP_K_RESULTS} results")

## 📄 Step 2: Document Loading & Processing 

Let's load our Flintstone API JSON file and convert it to readable text for better embedding quality.

In [ ]:
def json_to_text(data, indent=0):
    """
    🔄 Convert JSON to readable text format
    This helps with better chunking and embedding quality
    """
    result = []
    prefix = "  " * indent
    
    if isinstance(data, dict):
        for key, value in data.items():
            result.append(f"{prefix}{key}:")
            if isinstance(value, (dict, list)):
                result.append(json_to_text(value, indent + 1))
            else:
                result.append(f"{prefix}  {value}")
    elif isinstance(data, list):
        for item in data:
            if isinstance(item, (dict, list)):
                result.append(json_to_text(item, indent))
            else:
                result.append(f"{prefix}- {item}")
    
    return "\n".join(result)

def load_and_convert_document():
    """📖 Load the JSON document and convert to text"""
    print(f"📖 Loading document: {RAGConfig.INPUT_DOCUMENT}")
    
    # Load JSON file
    with open(RAGConfig.INPUT_DOCUMENT, 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    # Convert to text
    text_content = json_to_text(data)
    
    # Save converted text for inspection 🔍
    with open("converted_text_output.txt", "w", encoding='utf-8') as f:
        f.write("=" * 60 + "\n")
        f.write("CONVERTED JSON TO TEXT OUTPUT\n")
        f.write(f"Source: {RAGConfig.INPUT_DOCUMENT}\n")
        f.write("=" * 60 + "\n\n")
        f.write(text_content)
    
    print(f"✅ Document loaded: {len(text_content)} characters")
    print(f"💾 Converted text saved to: converted_text_output.txt")
    
    # Show first 500 characters as preview
    print("\n📋 Preview of converted text:")
    print("-" * 50)
    print(text_content[:500] + "..." if len(text_content) > 500 else text_content)
    
    return text_content

# Load and convert our document
document_text = load_and_convert_document()

## ✂️ Step 3: Text Chunking

Now let's break our document into smaller chunks for better retrieval. We'll use overlapping chunks to maintain context.

In [ ]:
def chunk_text(text: str) -> List[str]:
    """
    ✂️ Split text into overlapping chunks
    
    Why overlapping? To maintain context between chunks and avoid
    cutting important information in half.
    """
    chunks = []
    start = 0
    
    while start < len(text):
        # Get chunk of specified size
        end = start + RAGConfig.CHUNK_SIZE
        chunk = text[start:end]
        
        # If this isn't the last chunk, try to break at word boundary
        if end < len(text):
            last_space = chunk.rfind(' ')
            if last_space > 0:
                chunk = chunk[:last_space]
                end = start + last_space
        
        chunks.append(chunk.strip())
        
        # Move start position with overlap
        start = end - RAGConfig.CHUNK_OVERLAP
        
        # Prevent infinite loop if chunk overlap is too large
        if start >= end:
            start = end
    
    return chunks

def create_chunks_with_metadata(text: str) -> List[Dict[str, Any]]:
    """📋 Create chunks with metadata for tracking"""
    chunks = chunk_text(text)
    
    chunks_with_metadata = []
    for i, chunk in enumerate(chunks):
        chunk_data = {
            "id": i,
            "text": chunk,
            "length": len(chunk),
            "start_pos": i * (RAGConfig.CHUNK_SIZE - RAGConfig.CHUNK_OVERLAP)
        }
        chunks_with_metadata.append(chunk_data)
    
    # Save chunks for inspection 🔍
    with open("text_chunks_output.txt", "w", encoding='utf-8') as f:
        f.write("=" * 60 + "\n")
        f.write("TEXT CHUNKS OUTPUT\n")
        f.write(f"Total chunks: {len(chunks)}\n")
        f.write(f"Chunk size: {RAGConfig.CHUNK_SIZE} characters\n")
        f.write(f"Overlap: {RAGConfig.CHUNK_OVERLAP} characters\n")
        f.write("=" * 60 + "\n\n")
        
        for chunk_data in chunks_with_metadata:
            f.write(f"CHUNK {chunk_data['id'] + 1}:\n")
            f.write(f"ID: {chunk_data['id']}\n")
            f.write(f"Length: {chunk_data['length']} characters\n")
            f.write(f"Position: {chunk_data['start_pos']}-{chunk_data['start_pos'] + chunk_data['length']}\n")
            f.write("-" * 40 + "\n")
            f.write(chunk_data['text'][:200] + "..." if len(chunk_data['text']) > 200 else chunk_data['text'])
            f.write("\n" + "=" * 60 + "\n\n")
    
    print(f"✂️ Created {len(chunks)} chunks")
    print(f"💾 Chunks saved to: text_chunks_output.txt")
    print(f"📊 Average chunk size: {sum(len(c) for c in chunks) // len(chunks)} characters")
    
    return chunks_with_metadata

# Create our chunks
text_chunks = create_chunks_with_metadata(document_text)

## 🧮 Step 4: Generate Embeddings

Embeddings convert text into numerical vectors that capture semantic meaning. Similar texts will have similar vectors.

In [ ]:
def get_embedding(text: str) -> List[float]:
    """
    🧮 Get embedding vector for text using OpenAI
    
    What are embeddings?
    - Numerical representations of text meaning
    - Similar texts = similar vectors
    - Enable semantic search (not just keyword matching)
    """
    try:
        response = client.embeddings.create(
            input=text,
            model=RAGConfig.EMBEDDING_MODEL
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"❌ Error getting embedding: {e}")
        return []

def generate_embeddings_for_chunks(chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """🔄 Generate embeddings for all chunks"""
    print(f"🧮 Generating embeddings for {len(chunks)} chunks...")
    
    for i, chunk in enumerate(chunks):
        print(f"Processing chunk {i+1}/{len(chunks)}...", end=" ")
        
        embedding = get_embedding(chunk["text"])
        if embedding:
            chunk["embedding"] = embedding
            chunk["embedding_size"] = len(embedding)
            print("✅")
        else:
            print("❌")
            chunk["embedding"] = []
            chunk["embedding_size"] = 0
    
    # Show embedding statistics
    successful_embeddings = sum(1 for c in chunks if c.get("embedding"))
    if successful_embeddings > 0:
        embedding_size = chunks[0]["embedding_size"] if chunks[0].get("embedding") else 0
        print(f"\n📊 Embedding Statistics:")
        print(f"✅ Successful: {successful_embeddings}/{len(chunks)} chunks")
        print(f"📏 Vector size: {embedding_size} dimensions")
        print(f"🔢 Total vectors generated: {successful_embeddings}")
    
    return chunks

# Generate embeddings for all our chunks
embedded_chunks = generate_embeddings_for_chunks(text_chunks)

## 🗄️ Step 5: Create Vector Database

Let's save our embeddings to a JSON file - our simple vector database!

In [ ]:
def save_vector_database(chunks: List[Dict[str, Any]]):
    """
    🗄️ Save embeddings to JSON file (our vector database)
    
    Why JSON?
    - Simple and readable
    - No additional dependencies
    - Easy to inspect and debug
    """
    # Prepare data for JSON storage
    vector_data = {
        "metadata": {
            "source_document": RAGConfig.INPUT_DOCUMENT,
            "chunk_size": RAGConfig.CHUNK_SIZE,
            "chunk_overlap": RAGConfig.CHUNK_OVERLAP,
            "embedding_model": RAGConfig.EMBEDDING_MODEL,
            "total_chunks": len(chunks),
            "created_at": "2025-09-06"  # Current date
        },
        "chunks": []
    }
    
    # Add chunks (only successful embeddings)
    successful_chunks = [c for c in chunks if c.get("embedding")]
    for chunk in successful_chunks:
        vector_data["chunks"].append({
            "id": chunk["id"],
            "text": chunk["text"],
            "length": chunk["length"],
            "start_pos": chunk["start_pos"],
            "embedding": chunk["embedding"]
        })
    
    # Save to JSON file
    with open(RAGConfig.VECTOR_DATABASE_FILE, 'w', encoding='utf-8') as f:
        json.dump(vector_data, f, indent=2, ensure_ascii=False)
    
    print(f"🗄️ Vector database saved to: {RAGConfig.VECTOR_DATABASE_FILE}")
    print(f"📊 Stored {len(successful_chunks)} chunks with embeddings")
    print(f"💾 File size: {os.path.getsize(RAGConfig.VECTOR_DATABASE_FILE) / 1024:.1f} KB")
    
    return vector_data

def load_vector_database() -> Dict[str, Any]:
    """📖 Load vector database from JSON file"""
    try:
        with open(RAGConfig.VECTOR_DATABASE_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"📖 Loaded vector database: {len(data['chunks'])} chunks")
        return data
    except FileNotFoundError:
        print(f"❌ Vector database not found: {RAGConfig.VECTOR_DATABASE_FILE}")
        return {"chunks": [], "metadata": {}}

# Save our vector database
vector_database = save_vector_database(embedded_chunks)

# Show database structure
print(f"\n📋 Database Structure:")
print(f"📁 Metadata keys: {list(vector_database['metadata'].keys())}")
print(f"🔢 Chunks stored: {len(vector_database['chunks'])}")
if vector_database['chunks']:
    print(f"🧮 Embedding dimensions: {len(vector_database['chunks'][0]['embedding'])}")

## 🔍 Step 6: Similarity Search Implementation

Now the core of RAG - finding relevant chunks using cosine similarity!

In [ ]:
def cosine_similarity(vec1: List[float], vec2: List[float]) -> float:
    """
    🧮 Calculate cosine similarity between two vectors
    
    Cosine similarity measures the angle between vectors:
    - 1.0 = identical direction (most similar)
    - 0.0 = perpendicular (unrelated)
    - -1.0 = opposite direction (most dissimilar)
    """
    vec1_np = np.array(vec1)
    vec2_np = np.array(vec2)
    
    # Calculate dot product and magnitudes
    dot_product = np.dot(vec1_np, vec2_np)
    magnitude1 = np.linalg.norm(vec1_np)
    magnitude2 = np.linalg.norm(vec2_np)
    
    # Avoid division by zero
    if magnitude1 == 0 or magnitude2 == 0:
        return 0.0
    
    return dot_product / (magnitude1 * magnitude2)

def search_similar_chunks(question: str, vector_db: Dict[str, Any], top_k: int = None) -> List[Dict[str, Any]]:
    """
    🔍 Find most similar chunks to the question
    
    Process:
    1. Convert question to embedding
    2. Compare with all chunk embeddings using cosine similarity  
    3. Return top-k most similar chunks
    """
    if top_k is None:
        top_k = RAGConfig.TOP_K_RESULTS
    
    print(f"🔍 Searching for: '{question}'")
    
    # Get question embedding
    question_embedding = get_embedding(question)
    if not question_embedding:
        print("❌ Failed to get question embedding")
        return []
    
    # Calculate similarities with all chunks
    results = []
    for chunk in vector_db["chunks"]:
        if not chunk.get("embedding"):
            continue
            
        similarity = cosine_similarity(question_embedding, chunk["embedding"])
        results.append({
            "chunk_id": chunk["id"],
            "text": chunk["text"],
            "similarity": similarity,
            "length": chunk["length"]
        })
    
    # Sort by similarity (highest first) and take top-k
    results.sort(key=lambda x: x["similarity"], reverse=True)
    top_results = results[:top_k]
    
    print(f"📊 Found {len(results)} chunks, returning top {len(top_results)}")
    
    # Save retrieval results for inspection 🔍
    import time
    timestamp = int(time.time() * 10000) % 10000  # Last 4 digits for uniqueness
    filename = f"retrieval_results_{timestamp}.txt"
    
    with open(filename, "w", encoding='utf-8') as f:
        f.write("=" * 60 + "\\n")
        f.write("RETRIEVAL RESULTS\\n")
        f.write(f"Question: {question}\\n")
        f.write(f"Top-K results: {top_k}\\n")
        f.write("=" * 60 + "\\n\\n")
        
        for i, result in enumerate(top_results, 1):
            f.write(f"RETRIEVED CHUNK {i}:\\n")
            f.write(f"Chunk ID: {result['chunk_id']}\\n")
            f.write(f"Similarity Score: {result['similarity']:.6f}\\n")
            f.write(f"Text Length: {result['length']} characters\\n")
            f.write("-" * 40 + "\\n")
            f.write(result['text'])
            f.write("\\n" + "=" * 60 + "\\n\\n")
    
    print(f"💾 Retrieval results saved to: {filename}")
    
    return top_results

# Test the search function
test_question = "How do I authenticate with the Flintstone API?"
search_results = search_similar_chunks(test_question, vector_database)

print(f"\\n🎯 Top results for: '{test_question}'")
for i, result in enumerate(search_results, 1):
    print(f"\\n{i}. Similarity: {result['similarity']:.3f}")
    print(f"   Preview: {result['text'][:100]}...")

## 🤖 Step 7: Answer Generation

Finally, let's use the retrieved chunks to generate contextual answers!

In [ ]:
def generate_answer(question: str, context_chunks: List[Dict[str, Any]]) -> str:
    """
    🤖 Generate answer using retrieved context
    
    This combines the retrieved chunks with the user's question
    to generate a contextual, accurate response.
    """
    # Combine context from top chunks
    context = "\\n\\n".join([chunk["text"] for chunk in context_chunks])
    
    # Create prompt with context and question
    prompt = f"""You are a helpful assistant answering questions about the Flintstone API documentation.

Use the following context to answer the question. Be specific and accurate.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""
    
    try:
        response = client.chat.completions.create(
            model=RAGConfig.GENERATION_MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ],
            max_tokens=RAGConfig.MAX_TOKENS,
            temperature=RAGConfig.TEMPERATURE
        )
        
        answer = response.choices[0].message.content.strip()
        return answer
        
    except Exception as e:
        return f"❌ Error generating answer: {e}"

def ask_question(question: str, vector_db: Dict[str, Any]) -> Dict[str, Any]:
    """
    🎯 Complete RAG pipeline: Retrieve + Generate
    
    This is the main function that ties everything together:
    1. Search for relevant chunks
    2. Generate answer using those chunks
    3. Return complete response with metadata
    """
    print(f"\\n🎯 Processing question: '{question}'")
    print("=" * 50)
    
    # Step 1: Retrieve relevant chunks
    print("🔍 Retrieving relevant chunks...")
    retrieved_chunks = search_similar_chunks(question, vector_db)
    
    if not retrieved_chunks:
        return {
            "question": question,
            "answer": "❌ No relevant information found.",
            "retrieved_chunks": 0,
            "similarities": []
        }
    
    # Step 2: Generate answer
    print("🤖 Generating answer...")
    answer = generate_answer(question, retrieved_chunks)
    
    # Prepare response
    response = {
        "question": question,
        "answer": answer,
        "retrieved_chunks": len(retrieved_chunks),
        "similarities": [round(chunk["similarity"], 3) for chunk in retrieved_chunks]
    }
    
    print("✅ Answer generated!")
    print(f"📊 Used {len(retrieved_chunks)} chunks (similarities: {response['similarities']})")
    
    return response

# Test our complete RAG system!
test_questions = [
    "How do I authenticate with the Flintstone API?",
    "What are the rate limits?",
    "How do I get information about citizens?"
]

print("🚀 Testing Complete RAG System!")
print("=" * 60)

for question in test_questions:
    response = ask_question(question, vector_database)
    
    print(f"\\n❓ Q: {response['question']}")
    print(f"✅ A: {response['answer']}")
    print(f"📊 Chunks used: {response['retrieved_chunks']}, Similarities: {response['similarities']}")
    print("-" * 60)

## 🎮 Step 8: Interactive Exploration

Try asking your own questions! Use the function below to explore the RAG system.

In [ ]:
# 🎮 Try your own questions here!
# Change the question and run this cell

your_question = "What dinosaur services are available?"

# Ask the question
response = ask_question(your_question, vector_database)

# Display detailed results
print("🎯 DETAILED RAG RESPONSE")
print("=" * 50)
print(f"❓ Question: {response['question']}")
print(f"\\n✅ Answer:\\n{response['answer']}")
print(f"\\n📊 Retrieved {response['retrieved_chunks']} chunks")
print(f"🔢 Similarity scores: {response['similarities']}")
print("\\n💾 Check the generated files for detailed outputs:")
print("  • converted_text_output.txt - Original JSON converted to text")
print("  • text_chunks_output.txt - All text chunks")
print("  • retrieval_results_*.txt - Retrieved chunks with similarity scores")
print("  • vector_database.json - Complete vector database")

## 🎉 Congratulations! You Built RAG from Scratch!

### 🧠 **What You've Learned:**

1. **📄 Document Processing** - Converting JSON to searchable text
2. **✂️ Text Chunking** - Breaking documents into manageable pieces with overlap
3. **🧮 Embeddings** - Converting text to numerical vectors that capture meaning
4. **🗄️ Vector Database** - Storing embeddings in a simple JSON format
5. **🔍 Similarity Search** - Using cosine similarity to find relevant content
6. **🤖 Answer Generation** - Combining retrieval with AI to create responses
7. **🔬 Transparency** - Saving all outputs to files for inspection

### 🔧 **Key RAG Components:**

- **R**etrieval: Finding relevant chunks using vector similarity
- **A**ugmented: Enhancing the AI with retrieved context  
- **G**eneration: Creating answers based on relevant information

### 🚀 **Next Steps to Explore:**

1. **📚 Add More Documents** - Try different file formats (PDF, markdown, text)
2. **⚙️ Tune Parameters** - Experiment with chunk size, overlap, and top-k results
3. **🔄 Different Models** - Try other embedding models or GPT-4
4. **🎯 Better Chunking** - Implement semantic chunking or paragraph-based splitting
5. **📊 Evaluation** - Add metrics to measure retrieval and generation quality

### 💡 **Understanding the Magic:**

- **Embeddings** transform text into numbers that represent meaning
- **Cosine similarity** finds the "angle" between vectors (closer angle = more similar)
- **Context injection** gives the AI relevant information to generate better answers
- **Chunk overlap** ensures important information doesn't get split

You now understand the fundamentals of RAG and can build upon this foundation! 🎓